<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_12_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
!pip install neuralforecast -q
!pip install kaleido==0.2.1 -q
!pip install workalendar -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 79.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
   ━━━━━━

In [4]:
import pandas as pd
import numpy as np
import torch
import gc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from workalendar.europe import Russia
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
def save_predictions(ts, house_ids, y_true, y_pred, horizon_name, model_name, filename):
    df_pred = pd.DataFrame({
        'timestamp': ts,
        'house_id': house_ids,
        'y_true':y_true,
        'y_pred': y_pred,
        'horizon': horizon_name,
        'model': model_name,
    })
    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_pred = pd.concat([df_existing, df_pred], ignore_index=True)
    df_pred.to_csv(filename, index=False)

In [6]:
HORIZON_NAME = "4h"

In [7]:
HOUSE_META = {
    "house_1": {"n_flats": 383, "n_floors": 12},
    "house_2": {"n_flats": 191, "n_floors": 12},
    "house_3": {"n_flats": 124, "n_floors": 12},
    "house_4": {"n_flats": 263, "n_floors": 12},
    "house_5": {"n_flats": 127, "n_floors": 7},
    "house_6": {"n_flats": 497, "n_floors": 25},
    "house_7": {"n_flats": 471, "n_floors": 17},
    "house_8": {"n_flats": 171, "n_floors": 23},
}

df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()

def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h": 8,
    "8h": 16,
    "24h": 48,
    "7d": 336,
    "14d": 672,
    "1m": 1488,
}

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# нормализация per house только по train
tft_scalers = {}
dfs = []
for house in houses:
    train_mask = df["timestamp"] < df["timestamp"].iloc[train_end]
    scaler = MinMaxScaler()
    scaler.fit(df.loc[train_mask, [house]])
    tft_scalers[house] = scaler

    y_scaled = scaler.transform(df[[house]]).flatten()

    tmp = pd.DataFrame({
        "unique_id": house,
        "ds": df["timestamp"],
        "y": y_scaled,
    })
    dfs.append(tmp)

df_nf = pd.concat(dfs, ignore_index=True)
df_nf["y"] = df_nf.groupby("unique_id")["y"].transform(
    lambda x: x.interpolate(method="linear").ffill().bfill()
)

static_df = pd.DataFrame([
    {
        "unique_id": house,
        "n_flats": HOUSE_META[house]["n_flats"],
        "n_floors": HOUSE_META[house]["n_floors"],
    }
    for house in houses
])

df_val = df_nf[
    (df_nf["ds"] >= df["timestamp"].iloc[train_end])
    & (df_nf["ds"] < df["timestamp"].iloc[val_end])
]
df_test = df_nf[df_nf["ds"] >= df["timestamp"].iloc[val_end]]

val_size = len(df_val) // len(houses)
test_size = len(df_test) // len(houses)

print(f"Строк: {len(df)}, домов: {len(houses)}")
print(f"Val size: {val_size}, Test size: {test_size}")

Строк: 36260, домов: 8
Val size: 5439, Test size: 5439


In [9]:
torch.cuda.empty_cache()
gc.collect()

tft_results = {}
cv_dfs_tft = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc="TFT горизонты"):
    print(f"\nГоризонт {horizon_name} ({horizon_points} точек)")

    torch.cuda.empty_cache()
    gc.collect()

    nf = NeuralForecast(
        models=[
            TFT(
                h=horizon_points,
                input_size=max(336, horizon_points),
                hidden_size=32,
                n_head=2,
                max_steps=200,
                val_check_steps=50,
                early_stop_patience_steps=5,
                accelerator="gpu",
                devices=1,
            )
        ],
        freq="30min",
    )

    try:
        cv_df = nf.cross_validation(
            df=df_nf,
            static_df=static_df,
            val_size=val_size,
            test_size=test_size,
            n_windows=None,
        )

        cv_dfs_tft[horizon_name] = cv_df.copy()

        horizon_results = {}
        for house in houses:
            mask_h = cv_df["unique_id"] == house

            y_true_orig = tft_scalers[house].inverse_transform(
                cv_df.loc[mask_h, "y"].values.reshape(-1, 1)
            ).flatten()
            y_pred_orig = tft_scalers[house].inverse_transform(
                cv_df.loc[mask_h, "TFT"].values.reshape(-1, 1)
            ).flatten()

            metrics = evaluate(y_true_orig, y_pred_orig)
            horizon_results[house] = metrics

            # сохраняем последний тестовый период
            last_cutoff = cv_df.loc[mask_h, "cutoff"].max()
            mask_test = mask_h & (cv_df["cutoff"] == last_cutoff)

            y_true_test = tft_scalers[house].inverse_transform(
                cv_df.loc[mask_test, "y"].values.reshape(-1, 1)
            ).flatten()
            y_pred_test = tft_scalers[house].inverse_transform(
                cv_df.loc[mask_test, "TFT"].values.reshape(-1, 1)
            ).flatten()

            save_predictions(
                ts=cv_df.loc[mask_test, "ds"].values,
                house_ids=np.full(mask_test.sum(), house),
                y_true=y_true_test,
                y_pred=y_pred_test,
                horizon_name=horizon_name,
                model_name="TFT",
                filename="predictions_tft.csv",
            )

            print(f"  {house}: MAPE={metrics['MAPE']:.3f}%")

        tft_results[horizon_name] = horizon_results

        # сохраняем csv для графика
        cv_df[cv_df["unique_id"] == "house_1"].to_csv(
            f"cv_tft_{horizon_name}_house1.csv", index=False
        )

    except Exception as e:
        print(f"  Ошибка на горизонте {horizon_name}: {e}")
        tft_results[horizon_name] = {
            house: {"MAE": float("nan"), "RMSE": float("nan"), "MAPE": float("nan")}
            for house in houses
        }

    torch.cuda.empty_cache()
    gc.collect()

# сводная таблица
rows = []
for horizon_name in horizons.keys():
    for house in houses:
        metrics = tft_results[horizon_name][house]
        rows.append({
            "модель": "TFT",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_tft = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_tft.pivot(index="горизонт", columns="дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    pivot["Среднее"] = pivot.mean(axis=1).round(2)
    print(f"\nTFT - {metric}:")
    print(pivot.round(3).to_string())

df_tft.to_csv("results_tft.csv", index=False)

TFT горизонты:   0%|          | 0/6 [00:00<?, ?it/s]


Горизонт 4h (8 точек)


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MAE                      │      0 │ train │     0 │
│ 1 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 2 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 3 │ embedding               │ TFTEmbedding             │    128 │ train │     0 │
│ 4 │ temporal_encoder        │ TemporalCovariateEncoder │ 39.6 K │ train │     0 │
│ 5 │ temporal_fusion_decoder │ TemporalFusionDecoder    │ 17.0 K │ train │     0 │
│ 6 │ output_adapter          │ Linear                   │     33 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 56.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 56.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 88                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

  Ошибка на горизонте 1m: CUDA out of memory. Tried to allocate 67.57 GiB. GPU 0 has a total capacity of 39.49 GiB of which 21.04 GiB is free. Including non-PyTorch memory, this process has 18.44 GiB memory in use. Of the allocated memory 17.86 GiB is allocated by PyTorch, and 71.57 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


TFT горизонты: 100%|██████████| 6/6 [03:25<00:00, 34.19s/it]


TFT - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          4.540    2.234    1.544    3.165    2.209    6.711    5.990    2.717     3.64
8h          5.598    2.544    1.685    3.626    2.544    8.027    7.887    3.124     4.38
24h         5.725    2.605    1.694    3.837    2.470    7.909    7.653    3.083     4.37
7d         13.162    5.496    3.495    8.126    5.505   17.339   20.243    7.346    10.09
14d           NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN
1m            NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN

TFT - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          6.800    3.143    2.205    4.389    3.124    8.825    8.047    

In [10]:
torch.cuda.empty_cache()
gc.collect()

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
)

for i, (hn, hp) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    if hn not in cv_dfs_tft:
        continue

    house_cv = cv_dfs_tft[hn][
        cv_dfs_tft[hn]["unique_id"] == "house_1"
    ].reset_index(drop=True)

    cutoffs_sorted = sorted(house_cv["cutoff"].unique())
    mid_cutoff = cutoffs_sorted[len(cutoffs_sorted) // 2]
    subset = house_cv[
        house_cv["cutoff"] == mid_cutoff
    ].sort_values("ds").head(hp)

    y_true_plot = tft_scalers["house_1"].inverse_transform(
        subset["y"].values.reshape(-1, 1)
    ).flatten()
    y_pred_plot = tft_scalers["house_1"].inverse_transform(
        subset["TFT"].values.reshape(-1, 1)
    ).flatten()

    fig.add_trace(go.Scatter(
        x=subset["ds"], y=y_true_plot,
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=subset["ds"], y=y_pred_plot,
        mode="lines", name="прогноз TFT",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="TFT: прогнозные и фактические значения по всем горизонтам (дом 1)",
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9)),
)
fig.show()
fig.write_image("36_tft_forecast_all_horizons.png", scale=2)